In [11]:
# acquisition_bias.py

import numpy as np
import pandas as pd
import random
import json

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    log_loss,
)

In [12]:
# 1. Reproducibility

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [13]:
# 2. Load Dataset

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [14]:
# 3. Train-Test Split (UNCHANGED)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

In [15]:
# 4. Introduce Acquisition Bias

feature = "worst perimeter"

threshold = X_train[feature].quantile(0.80)

mask = X_train[feature] <= threshold

X_train_biased = X_train[mask]
y_train_biased = y_train[mask]

print(f"Original training size: {len(X_train)}")
print(f"Biased training size: {len(X_train_biased)}")

Original training size: 455
Biased training size: 364


In [16]:
# 5. Preprocessing (REFIT ON BIASED DATA)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_biased)
X_test_scaled = scaler.transform(X_test)

In [17]:
# 6. Models

models = {
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=SEED),
}

results = {}

In [18]:
# 7. Training & Evaluation

for name, model in models.items():
    model.fit(X_train_scaled, y_train_biased)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    loss = log_loss(y_test, y_prob)

    results[name] = {
        "accuracy": acc,
        "f1_score": f1,
        "roc_auc": roc,
        "log_loss": loss,
    }

    print(f"\nModel: {name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC AUC: {roc:.4f}")
    print(f"Log Loss: {loss:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: LogisticRegression
Accuracy: 0.9825
F1 Score: 0.9861
ROC AUC: 0.9964
Log Loss: 0.0718
Confusion Matrix:
[[41  1]
 [ 1 71]]

Model: RandomForest
Accuracy: 0.9474
F1 Score: 0.9583
ROC AUC: 0.9954
Log Loss: 0.1180
Confusion Matrix:
[[39  3]
 [ 3 69]]


In [19]:
# 8. Save Results

with open("acquisition_bias_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nAcquisition bias experiment completed.")


Acquisition bias experiment completed.
